# FNO input preparation, Optuna optimization, and training

This notebook replaces the older FNO input, Optuna, and final-training scripts. It prepares point inventories for ASCAT and SMAP, runs the actual hyperparameter search, and trains the ten-seed ensemble.

The experiment retains seven static predictors, past-only windows of 32 valid satellite observations with scaled time gaps, fixed temporal validation/holdout dates, an 80/20 spatial point split, and a validation objective weighted 70% toward spatial transfer. It does not generate figures, parameter text files, manifests, or version records. Each product is saved as one prediction-ready bundle containing only its hyperparameters, scaler, and trained weights.

In [ ]:
import os

from config import configure_runtime

configure_runtime()

import pandas as pd
import torch
from IPython.display import display

## 1. Paths and reusable imports

In [ ]:
from config import (
    FNO_TRAINING_WORKERS,
    base_FP,
    cpuserver_data,
    das_FP,
    george_FP,
    nas_FP,
)
from FNO.inputs import generate_input_csv
from FNO.settings import (
    AREAS,
    CONTEXT_END_DATE,
    CONTEXT_START_DATE,
    MODEL_HOLDOUT_START_DATE,
    MODEL_SEEDS,
    PRODUCTS,
    STATIC_FEATURES,
    VALIDATION_START_DATE,
    WINDOW_SIZE,
)
from FNO.training import (
    optimize_hyperparameters,
    save_ensemble_bundle,
    train_ensemble,
)

from config import figures_FP, results_FP


## 2. Scientific and runtime settings

The calendar is 2015-04-01 through 2023-12-31. Training targets stop before 2021-06-15, temporal validation spans 2021-06-15 through 2022-04-20, and later observations remain held out. Static features are fitted only on the deterministic 80% spatial-training subset. Optuna uses seed 42; final training uses all ten listed seeds.

In [ ]:
DATA_PRODUCTS = PRODUCTS
INPUT_AREAS = AREAS
MIN_VALID_OBSERVATIONS = 100
N_OPTUNA_TRIALS = 200
OPTUNA_EPOCHS = 50
FINAL_TRAINING_EPOCHS = 500
FINAL_MODEL_SEEDS = MODEL_SEEDS
TRAINING_WORKERS = FNO_TRAINING_WORKERS
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# False keeps only the two final prediction-ready model bundles.
SAVE_OPTUNA_DATABASES = False

print(f"Device: {DEVICE}")
print(f"Final-training workers: {TRAINING_WORKERS}")
print(f"Static features ({len(STATIC_FEATURES)}): {list(STATIC_FEATURES)}")
print(f"Observation window: {WINDOW_SIZE} valid SSM events")
print(
    f"Calendar/splits: {CONTEXT_START_DATE}..{CONTEXT_END_DATE}; "
    f"validation={VALIDATION_START_DATE}; holdout={MODEL_HOLDOUT_START_DATE}"
)

In [ ]:
ISMN_RESULT_FP = os.path.join(results_FP, "ISMN")
MODEL_INPUT_FILE = os.path.join(results_FP, "CONUS_Prediction", "Model_input_static_eqd_010.nc"
)
FNO_RESULT_FP = os.path.join(results_FP, "FNO")
TRAIN_RESULT_FP = os.path.join(FNO_RESULT_FP, "Train")
os.makedirs(TRAIN_RESULT_FP, exist_ok=True)

INPUT_CSV = {
    (area, product): os.path.join(
        FNO_RESULT_FP, area, f"FNO_input_{product}.csv"
    )
    for area in INPUT_AREAS
    for product in DATA_PRODUCTS
}
BUNDLE_FILE = {
    product: os.path.join(TRAIN_RESULT_FP, f"FNO_{product}_ensemble.pt")
    for product in DATA_PRODUCTS
}

if not os.path.exists(MODEL_INPUT_FILE):
    raise FileNotFoundError(f"Missing static model input: {MODEL_INPUT_FILE}")

## 3. Prepare point/static input CSV files

The CSVs are compact pixel inventories. Dynamic SSM and observed RZSM remain in the point NetCDF files produced by `ISMN_preprocessing.ipynb`.

In [ ]:
input_summaries = []

for area in INPUT_AREAS:
    for product in DATA_PRODUCTS:
        input_frame = generate_input_csv(
            product=product,
            area=area,
            model_input_file=MODEL_INPUT_FILE,
            ismn_root=ISMN_RESULT_FP,
            output_file=INPUT_CSV[(area, product)],
            min_valid_observations=MIN_VALID_OBSERVATIONS,
        )
        input_summaries.append(
            {
                "area": area,
                "product": product,
                "pixels": 0 if input_frame is None else len(input_frame),
                "file": INPUT_CSV[(area, product)],
            }
        )

display(pd.DataFrame(input_summaries))

## 4. Optimize hyperparameters

This is the expensive Optuna stage. By default its studies remain in memory and create no database or parameter-report files. Set `SAVE_OPTUNA_DATABASES = True` only when resumable studies are needed.

In [ ]:
optimization_results = {}

for product in DATA_PRODUCTS:
    storage = None
    if SAVE_OPTUNA_DATABASES:
        database_file = os.path.join(TRAIN_RESULT_FP, f"FNO_{product}_optuna.db")
        storage = f"sqlite:///{database_file}"

    best_parameters, scaler, study = optimize_hyperparameters(
        product=product,
        input_csv=INPUT_CSV[("Train", product)],
        ismn_root=ISMN_RESULT_FP,
        n_trials=N_OPTUNA_TRIALS,
        num_epochs=OPTUNA_EPOCHS,
        device=DEVICE,
        storage=storage,
    )
    optimization_results[product] = {
        "best_parameters": best_parameters,
        "scaler": scaler,
        "study": study,
    }
    print(product, best_parameters)

## 5. Train and save the final ten-seed ensembles

Each output bundle is the only trained-model artifact needed by `FNO_prediction.ipynb`. Loss-curve figures and historical version/provenance records are not produced.

In [ ]:
training_summaries = []

for product in DATA_PRODUCTS:
    optimized = optimization_results[product]
    ensemble = train_ensemble(
        product=product,
        input_csv=INPUT_CSV[("Train", product)],
        ismn_root=ISMN_RESULT_FP,
        scaler=optimized["scaler"],
        hyperparameters=optimized["best_parameters"],
        seeds=FINAL_MODEL_SEEDS,
        device=DEVICE,
        num_epochs=FINAL_TRAINING_EPOCHS,
        workers=TRAINING_WORKERS,
    )
    save_ensemble_bundle(
        output_file=BUNDLE_FILE[product],
        ensemble=ensemble,
        scaler=optimized["scaler"],
        hyperparameters=optimized["best_parameters"],
    )
    training_summaries.append(
        {
            "product": product,
            "models": len(ensemble),
            "bundle": BUNDLE_FILE[product],
        }
    )

display(pd.DataFrame(training_summaries))